# Breakdown

The large-scale evaluation dataset (DL) of 841 CVEs consists of 440 OSS; including (156) web applications/backend, (147) libraries/frameworks, (28) CLI tools/binaries, (25) cloud/DevOps, (18) embedded/networking, (17) OS/runtime, (16) AI/ML platforms, (13) desktop applications, (7) blockchain/crypto, (7) mobile applications/SDK, and (6) security tools. 

Meanwhile, the majority of CVEs (in DL) fall under these vulnerability categories; (~29%) injection and improper neutralization, (~11%) authorization and authentication, (~8%) resource management,  (~6%) information exposure, (~3%) memory safety, (~3%) request forgery, etc (we will add more detailed breakdown in the appendix).

# Analysis of Time/Cost Overrun Failures

In [1]:
import os
import re
import json
import pandas as pd
from openai import OpenAI

In [14]:
cves = {
    'exp-1': [],
    'exp-2': [],
    'exp-3': []
}
for exp_dir in ['exp-1', 'exp-2', 'exp-3']:
    results_file = f'large_scale/{exp_dir}/results.csv'
    df = pd.read_csv(results_file)
    for index, row in df.iterrows():
        if row['SUCCESS'] == False and ('Cost' in row['REASON'] or 'Time' in row['REASON']):
            cves[exp_dir].append(row['CVE'])

cves_1, cves_2, cves_3 = cves['exp-1'], cves['exp-2'], cves['exp-3']
len(cves_1), len(cves_2), len(cves_3)

(242, 231, 198)

In [17]:
# cves that had cost or time overruns in all three experiments
cves_all_overrun = set(cves_1) & set(cves_2) & set(cves_3)
print(f"CVE that had cost or time overruns in all three experiments: {len(cves_all_overrun)}")

# cves that had cost or time overruns in cves_1 but became successful in cves_2
cves_1_overrun = set(cves_1) - set(cves_2)
print(f"CVE that had cost or time overruns in cves_1 but became successful in cves_2: {len(cves_1_overrun)}")

# cves that had cost or time overruns in cves_1 but became successful in cves_3
cves_1_overrun_but_success_3 = set(cves_1) - set(cves_3)
print(f"CVE that had cost or time overruns in cves_1 but became successful in cves_3: {len(cves_1_overrun_but_success_3)}")

# cves that had cost or time overruns in cves_2 but became successful in cves_3
cves_2_overrun_but_success_3 = set(cves_2) - set(cves_3)
print(f"CVE that had cost or time overruns in cves_2 but became successful in cves_3: {len(cves_2_overrun_but_success_3)}")

# cves that had cost or time overruns in cves_2 but became successful in cves_1
cves_2_overrun_but_success_1 = set(cves_2) - set(cves_1)
print(f"CVE that had cost or time overruns in cves_2 but became successful in cves_1: {len(cves_2_overrun_but_success_1)}")

CVE that had cost or time overruns in all three experiments: 74
CVE that had cost or time overruns in cves_1 but became successful in cves_2: 111
CVE that had cost or time overruns in cves_1 but became successful in cves_3: 127
CVE that had cost or time overruns in cves_2 but became successful in cves_3: 106
CVE that had cost or time overruns in cves_2 but became successful in cves_1: 100


In [ ]:
# total cost overruns and timeouts
for exp in ['exp-1', 'exp-2', 'exp-3']:
    print(f"Analyzing {exp}...")
    cost_overrun_cves = []
    timeout_cves = []
    df = pd.read_csv(f'large_scale/{exp}/results.csv')
    for index, row in df.iterrows():
        if row['CVE'] in cves_3 and row['SUCCESS'] == False:
            if 'Cost' in row['REASON']:
                cost_overrun_cves.append(row)
            elif 'Time' in row['REASON']:
                timeout_cves.append(row)

    print(f"Total cost overruns: {len(cost_overrun_cves)}")
    print(f"Total timeouts: {len(timeout_cves)}")

Analyzing exp-1...
Total cost overruns: 101
Total timeouts: 14
Analyzing exp-2...
Total cost overruns: 107
Total timeouts: 18
Analyzing exp-3...
Total cost overruns: 156
Total timeouts: 42


### Timeouts

In [126]:
# cves that had time overruns in cves_3 and what phase they failed in
timeout_reason = {
    'project_build': [],
    'exploit_build': [],
    'verifier_build': [],
    'unknown': []
}
total_timeouts = 0
df = pd.read_csv('large_scale/exp-3/results.csv')
for index, row in df.iterrows():
    if row['CVE'] in cves_3 and row['SUCCESS'] == False:
        if 'Time' in row['REASON']:
            total_timeouts += 1
            if 'phase: project_build' in row['REASON']:
                timeout_reason['project_build'].append(row['CVE'])
            elif 'phase: exploit_build' in row['REASON']:
                timeout_reason['exploit_build'].append(row['CVE'])
            elif 'phase: verifier_build' in row['REASON']:
                timeout_reason['verifier_build'].append(row['CVE'])
            else:
                timeout_reason['unknown'].append(row['CVE'])

print(f"Time overruns by phase:")
print(f"  Builder (project_build): {len(timeout_reason['project_build'])} ({len(timeout_reason['project_build'])/total_timeouts*100:.1f}%)")
print(f"  Exploiter (exploit_build): {len(timeout_reason['exploit_build'])} ({len(timeout_reason['exploit_build'])/total_timeouts*100:.1f}%)")
print(f"  CTF Verifier (verifier_build): {len(timeout_reason['verifier_build'])} ({len(timeout_reason['verifier_build'])/total_timeouts*100:.1f}%)")
print(f"  Unknown: {len(timeout_reason['unknown'])} ({len(timeout_reason['unknown'])/total_timeouts*100:.1f}%)")

Time overruns by phase:
  Builder (project_build): 11 (26.2%)
  Exploiter (exploit_build): 9 (21.4%)
  CTF Verifier (verifier_build): 20 (47.6%)
  Unknown: 2 (4.8%)


In [52]:
# what type of projects were in each category of time overruns
data = json.load(open('data.json'))
project_types = json.load(open('project_types.json'))
for reason in timeout_reason:
    print(f"{reason}:")
    types_count = {}
    for cve in timeout_reason[reason]:
        patch_url = data[cve]['patch_commits'][0]['url'].split('/')
        project_name = patch_url[-4] + '/' + patch_url[-3]
        project_type = [pt for pt in project_types if project_name in project_types[pt]][0]
        types_count[project_type] = types_count.get(project_type, 0) + 1
    for project_type in sorted(types_count, key=lambda x: types_count[x], reverse=True):
        print(f"  {project_type}: {types_count[project_type]}")
    print()

project_build:
  Web Application/Backend: 6
  Library/Framework: 3
  CLI Tool/Utility: 1
  AI/ML Platform: 1

exploit_build:
  Desktop Application: 4
  Web Application/Backend: 3
  Operating System/Runtime: 1
  Library/Framework: 1

verifier_build:
  Web Application/Backend: 11
  CLI Tool/Utility: 3
  Library/Framework: 2
  Desktop Application: 2
  Blockchain/Crypto/FinTech: 1
  Operating System/Runtime: 1

unknown:
  Web Application/Backend: 2



In [78]:
# overviews of the projects in each category of time overruns
overviews = {}
project_names = {}
for reason in list(timeout_reason.keys())[2:3]:
    print(f"{reason}:")
    for cve in timeout_reason[reason]:
        if 'patch_commits' not in data[cve] or not data[cve]['patch_commits']:
            continue
        patch_url = data[cve]['patch_commits'][0]['url'].split('/')
        project_name = patch_url[-4] + '/' + patch_url[-3]
        project_names[project_name] = project_names.get(project_name, 0) + 1
        if project_name not in overviews:
            pre_req_path = f'large_scale/exp-3/{cve}/conversations/pre_req_builder.json'
            overview = json.load(open(pre_req_path))['overview']
            overviews[project_name] = overview
open('time_overrun_overviews.json', 'w').write(json.dumps(overviews, indent=4))
project_names

verifier_build:


{'liske/needrestart': 1,
 'gohugoio/hugo': 1,
 'enchant97/note-mark': 1,
 'charmbracelet/soft-serve': 1,
 'phpipam/phpipam': 1,
 'danny-avila/librechat': 2,
 'HabitRPG/habitica': 1,
 'evmos/evmos': 1,
 'cvat-ai/cvat': 1,
 'lobehub/lobe-chat': 1,
 'rack/rack': 1,
 'GitoxideLabs/gitoxide': 1,
 'directus/directus': 2,
 'nexryai/concorde': 1,
 'ecki/net-tools': 1,
 'pbatard/rufus': 1,
 'bep/imagemeta': 1,
 'vim/vim': 1}

In [ ]:
# what type of advisories were in each category of time overruns
data = json.load(open('data.json'))
effectiveness = {
    'project_build': {'effective': [], 'ineffective': []},
    'exploit_build': {'effective': [], 'ineffective': []},
    'verifier_build': {'effective': [], 'ineffective': []}
}
for cve in timeout_reason['project_build']:
    if 'sec_adv' in data[cve] and data[cve]['sec_adv']:
        if any(adv['effective'] for adv in data[cve]['sec_adv']):
            effectiveness['project_build']['effective'].append(cve)
        else:
            effectiveness['project_build']['ineffective'].append(cve)
    else:
        effectiveness['project_build']['ineffective'].append(cve)

for cve in timeout_reason['exploit_build']:
    if 'sec_adv' in data[cve] and data[cve]['sec_adv']:
        if any(adv['effective'] for adv in data[cve]['sec_adv']):
            effectiveness['exploit_build']['effective'].append(cve)
        else:
            effectiveness['exploit_build']['ineffective'].append(cve)
    else:
        effectiveness['exploit_build']['ineffective'].append(cve)

for cve in timeout_reason['verifier_build']:
    if 'sec_adv' in data[cve] and data[cve]['sec_adv']:
        if any(adv['effective'] for adv in data[cve]['sec_adv']):
            effectiveness['verifier_build']['effective'].append(cve)
        else:
            effectiveness['verifier_build']['ineffective'].append(cve)

print(f"Project Build: {len(effectiveness['project_build']['effective'])} effective, {len(effectiveness['project_build']['ineffective'])} ineffective")
print(f"Exploit Build: {len(effectiveness['exploit_build']['effective'])} effective, {len(effectiveness['exploit_build']['ineffective'])} ineffective")
print(f"Verifier Build: {len(effectiveness['verifier_build']['effective'])} effective, {len(effectiveness['verifier_build']['ineffective'])} ineffective")

Project Build: 4 effective, 7 ineffective
Exploit Build: 5 effective, 4 ineffective
Verifier Build: 11 effective, 8 ineffective


In [73]:
# what type of CWEs were in each category of time overruns
data = json.load(open('data.json'))
for reason in timeout_reason:
    print(f"{reason}:")
    cwes = {}
    for cve in timeout_reason[reason]:
        if 'cwe' in data[cve] and data[cve]['cwe']:
            for cwe in data[cve]['cwe']:
                cwes[cwe['id']] = cwes.get(cwe['id'], 0) + 1
    for cwe in sorted(cwes, key=lambda x: cwes[x], reverse=True):
        print(f"  {cwe}: {cwes[cwe]}")
    print()

project_build:
  CWE-770: 2
  CWE-400: 1
  CWE-305: 1
  CWE-476: 1
  CWE-841: 1
  CWE-601: 1
  CWE-918: 1
  CWE-79: 1
  CWE-269: 1
  CWE-532: 1
  CWE-306: 1
  CWE-77: 1

exploit_build:
  CWE-416: 2
  CWE-89: 1
  CWE-79: 1
  CWE-524: 1
  CWE-918: 1
  CWE-122: 1
  n/a: 1
  CWE-601: 1

verifier_build:
  CWE-79: 5
  n/a: 1
  CWE-78: 1
  CWE-915: 1
  CWE-284: 1
  CWE-682: 1
  CWE-81: 1
  CWE-918: 1
  CWE-1333: 1
  CWE-328: 1
  CWE-672: 1
  CWE-200: 1
  CWE-613: 1
  CWE-20: 1
  CWE-121: 1
  CWE-426: 1
  CWE-427: 1
  CWE-770: 1
  CWE-122: 1

unknown:
  CWE-20: 1
  CWE-862: 1



### Cost Overruns

In [127]:
# cves that had cost overruns in cves_3 and what phase they failed in
cost_overrun_reason = {
    'project_build': [],
    'exploit_build': [],
    'verifier_build': [],
    'unknown': []
}
total_cost_overruns = 0
df = pd.read_csv('large_scale/exp-3/results.csv')
builder_mark = "🏭 Repository Builder"
exploiter_mark = "🚀 Running Exploiter ..."
verifier_mark = "🛡️ CTF Verifier"
for index, row in df.iterrows():
    if row['CVE'] in cves_3 and row['SUCCESS'] == False:
        if 'Cost' in row['REASON']:
            total_cost_overruns += 1
            cve_log_file = f'large_scale/exp-3/{row["CVE"]}/log.txt'
            if os.path.exists(cve_log_file):
                with open(cve_log_file, 'r') as f:
                    log_content = f.read()
                    if builder_mark in log_content and exploiter_mark in log_content and verifier_mark in log_content:
                        cost_overrun_reason['verifier_build'].append(row['CVE'])
                    elif builder_mark in log_content and exploiter_mark in log_content:
                        cost_overrun_reason['exploit_build'].append(row['CVE'])
                    elif builder_mark in log_content:
                        cost_overrun_reason['project_build'].append(row['CVE'])
                    else:
                        cost_overrun_reason['unknown'].append(row['CVE'])

print(f"Cost overruns by phase:")
print(f"  Builder (project_build): {len(cost_overrun_reason['project_build'])} ({len(cost_overrun_reason['project_build'])/total_cost_overruns*100:.1f}%)")
print(f"  Exploiter (exploit_build): {len(cost_overrun_reason['exploit_build'])} ({len(cost_overrun_reason['exploit_build'])/total_cost_overruns*100:.1f}%)")
print(f"  CTF Verifier (verifier_build): {len(cost_overrun_reason['verifier_build'])} ({len(cost_overrun_reason['verifier_build'])/total_cost_overruns*100:.1f}%)")
print(f"  Unknown: {len(cost_overrun_reason['unknown'])} ({len(cost_overrun_reason['unknown'])/total_cost_overruns*100:.1f}%)")

Cost overruns by phase:
  Builder (project_build): 65 (41.7%)
  Exploiter (exploit_build): 91 (58.3%)
  CTF Verifier (verifier_build): 0 (0.0%)
  Unknown: 0 (0.0%)


In [55]:
# what type of projects were in each category of cost overruns
data = json.load(open('data.json'))
project_types = json.load(open('project_types.json'))
for reason in cost_overrun_reason:
    print(f"{reason}:")
    types_count = {}
    for cve in cost_overrun_reason[reason]:
        if 'patch_commits' not in data[cve] or not data[cve]['patch_commits']:
            continue
        patch_url = data[cve]['patch_commits'][0]['url'].split('/')
        project_name = patch_url[-4] + '/' + patch_url[-3]
        project_type = [pt for pt in project_types if project_name in project_types[pt]][0]
        types_count[project_type] = types_count.get(project_type, 0) + 1
    for project_type in sorted(types_count, key=lambda x: types_count[x], reverse=True):
        print(f"  {project_type}: {types_count[project_type]}")
    print()

project_build:
  Web Application/Backend: 48
  Embedded/Networking: 4
  Library/Framework: 2
  Operating System/Runtime: 2
  Cloud/DevOps/Orchestration: 2
  AI/ML Platform: 2
  Mobile Application/SDK: 2
  Desktop Application: 1
  CLI Tool/Utility: 1

exploit_build:
  Web Application/Backend: 40
  Library/Framework: 17
  AI/ML Platform: 7
  Desktop Application: 5
  CLI Tool/Utility: 5
  Embedded/Networking: 5
  Blockchain/Crypto/FinTech: 4
  Security Tool/Server: 3
  Operating System/Runtime: 2
  Cloud/DevOps/Orchestration: 2
  Mobile Application/SDK: 1

verifier_build:

unknown:



In [82]:
# overviews of the web applications in failed categories
overviews = {}
project_names = {}
for cve in cost_overrun_reason['project_build']:
    if 'patch_commits' not in data[cve] or not data[cve]['patch_commits']:
        continue
    patch_url = data[cve]['patch_commits'][0]['url'].split('/')
    project_name = patch_url[-4] + '/' + patch_url[-3]
    if project_name not in project_types['Web Application/Backend']:
        continue
    project_names[project_name] = project_names.get(project_name, 0) + 1
    if project_name not in overviews:
        pre_req_path = f'large_scale/exp-3/{cve}/conversations/pre_req_builder.json'
        overview = json.load(open(pre_req_path))['overview']
        overviews[project_name] = overview
open('project_build_overviews.json', 'w').write(json.dumps(overviews, indent=4))
project_names

{'parisneo/lollms-webui': 7,
 'discourse/discourse': 12,
 'yugabyte/yugabyte-db': 3,
 'dataease/dataease': 3,
 'theonedev/onedev': 1,
 'zitadel/zitadel': 3,
 'decidim/decidim': 4,
 'berriai/litellm': 2,
 'OpenC3/cosmos': 2,
 'nextcloud/server': 1,
 'pimcore/pimcore': 1,
 'filamentphp/filament': 1,
 'bigbluebutton/bigbluebutton': 1,
 'open-webui/open-webui': 2,
 'oxyno-zeta/s3-proxy': 1,
 'mastodon/mastodon': 1,
 'apollographql/router': 1,
 'dnnsoftware/Dnn.Platform': 2}

In [ ]:
# what type of advisories were in each category of cost overruns
data = json.load(open('data.json'))
effectiveness = {
    'project_build': {'effective': [], 'ineffective': []},
    'exploit_build': {'effective': [], 'ineffective': []},
    'verifier_build': {'effective': [], 'ineffective': []}
}
for cve in cost_overrun_reason['project_build']:
    if 'sec_adv' in data[cve] and data[cve]['sec_adv']:
        if any(adv['effective'] for adv in data[cve]['sec_adv']):
            effectiveness['project_build']['effective'].append(cve)
        else:
            effectiveness['project_build']['ineffective'].append(cve)
    else:
        effectiveness['project_build']['ineffective'].append(cve)

for cve in cost_overrun_reason['exploit_build']:
    if 'sec_adv' in data[cve] and data[cve]['sec_adv']:
        if any(adv['effective'] for adv in data[cve]['sec_adv']):
            effectiveness['exploit_build']['effective'].append(cve)
        else:
            effectiveness['exploit_build']['ineffective'].append(cve)
    else:
        effectiveness['exploit_build']['ineffective'].append(cve)

print(f"Project Build: {len(effectiveness['project_build']['effective'])} effective, {len(effectiveness['project_build']['ineffective'])} ineffective")
print(f"Exploit Build: {len(effectiveness['exploit_build']['effective'])} effective, {len(effectiveness['exploit_build']['ineffective'])} ineffective")

Project Build: 25 effective, 40 ineffective
Exploit Build: 37 effective, 54 ineffective


In [80]:
# what type of CWEs were in each category of cost overruns
data = json.load(open('data.json'))
for reason in cost_overrun_reason:
    print(f"{reason}:")
    cwes = {}
    for cve in cost_overrun_reason[reason]:
        if 'cwe' in data[cve] and data[cve]['cwe']:
            for cwe in data[cve]['cwe']:
                cwes[cwe['id']] = cwes.get(cwe['id'], 0) + 1
    for cwe in sorted(cwes, key=lambda x: cwes[x], reverse=True):
        print(f"  {cwe}: {cwes[cwe]}")
    print()

project_build:
  CWE-79: 12
  CWE-400: 5
  CWE-200: 4
  CWE-22: 3
  CWE-20: 3
  CWE-532: 3
  CWE-918: 3
  CWE-285: 3
  CWE-203: 2
  CWE-770: 2
  CWE-502: 2
  CWE-284: 2
  CWE-352: 1
  CWE-36: 1
  CWE-29: 1
  CWE-94: 1
  CWE-303: 1
  CWE-305: 1
  CWE-190: 1
  CWE-187: 1
  CWE-444: 1
  CWE-863: 1
  CWE-312: 1
  CWE-798: 1
  CWE-476: 1
  CWE-287: 1
  CWE-306: 1
  CWE-862: 1
  CWE-77: 1
  CWE-639: 1
  CWE-74: 1
  CWE-1021: 1
  CWE-416: 1
  CWE-434: 1
  CWE-204: 1
  CWE-362: 1
  CWE-269: 1
  CWE-488: 1
  CWE-926: 1
  CWE-119: 1
  CWE-804: 1

exploit_build:
  CWE-79: 11
  CWE-22: 9
  CWE-20: 5
  CWE-94: 5
  CWE-416: 4
  CWE-918: 4
  CWE-78: 4
  CWE-285: 3
  CWE-284: 3
  CWE-269: 2
  CWE-502: 2
  CWE-248: 2
  CWE-61: 2
  CWE-122: 2
  CWE-200: 2
  CWE-532: 2
  CWE-639: 2
  CWE-287: 2
  CWE-472: 2
  CWE-306: 1
  CWE-29: 1
  CWE-184: 1
  CWE-400: 1
  CWE-328: 1
  CWE-327: 1
  CWE-115: 1
  CWE-23: 1
  CWE-299: 1
  CWE-924: 1
  CWE-670: 1
  CWE-362: 1
  CWE-1236: 1
  CWE-117: 1
  CWE-770: 1
  CWE-

### Other Failures

In [ ]:
# types of projects overall
project_types = json.load(open('project_types.json'))
project_names = list(project_types.values())
project_names = [item for sublist in project_names for item in sublist]
for project_type in project_types:
    print(f"{project_type}: {len(project_types[project_type])} ({len(project_types[project_type])/len(project_names)*100:.1f}%)")

AI/ML Platform: 16 (3.7%)
Web Application/Backend: 156 (35.6%)
Library/Framework: 147 (33.6%)
Cloud/DevOps/Orchestration: 25 (5.7%)
Operating System/Runtime: 15 (3.4%)
Desktop Application: 13 (3.0%)
CLI Tool/Utility: 28 (6.4%)
Blockchain/Crypto/FinTech: 7 (1.6%)
Mobile Application/SDK: 7 (1.6%)
Embedded/Networking: 18 (4.1%)
Security Tool/Server: 6 (1.4%)


In [90]:
# cves distribution by project type
data = json.load(open('data.json'))
project_types = json.load(open('project_types.json'))
project_cve_map = {pt: 0 for pt in project_types}
for cve in data:
    if 'patch_commits' not in data[cve] or not data[cve]['patch_commits']:
        continue
    patch_url = data[cve]['patch_commits'][0]['url'].split('/')
    project_name = patch_url[-4] + '/' + patch_url[-3]
    for pt in project_types:
        if project_name in project_types[pt]:
            project_cve_map[pt] += 1
            # break
for project_type in project_cve_map:
    print(f"{project_type}: {project_cve_map[project_type]} ({project_cve_map[project_type]/len(data)*100:.1f}%)")

AI/ML Platform: 38 (4.5%)
Web Application/Backend: 383 (45.5%)
Library/Framework: 224 (26.6%)
Cloud/DevOps/Orchestration: 33 (3.9%)
Operating System/Runtime: 31 (3.7%)
Desktop Application: 34 (4.0%)
CLI Tool/Utility: 37 (4.4%)
Blockchain/Crypto/FinTech: 13 (1.5%)
Mobile Application/SDK: 7 (0.8%)
Embedded/Networking: 27 (3.2%)
Security Tool/Server: 12 (1.4%)


In [84]:
# types of CWEs overall
data = json.load(open('data.json'))
cwes = {}
total_cwes = 0
for cve in data:
    if 'cwe' in data[cve] and data[cve]['cwe']:
        for cwe in data[cve]['cwe']:
            total_cwes += 1
            cwes[cwe['id']] = cwes.get(cwe['id'], 0) + 1
for cwe in sorted(cwes, key=lambda x: cwes[x], reverse=True):
    print(f"  {cwe}: {cwes[cwe]} ({cwes[cwe]/total_cwes*100:.1f}%)")
print()

  CWE-79: 142 (14.6%)
  CWE-200: 43 (4.4%)
  CWE-22: 37 (3.8%)
  CWE-284: 36 (3.7%)
  CWE-400: 36 (3.7%)
  CWE-20: 29 (3.0%)
  CWE-770: 24 (2.5%)
  CWE-918: 22 (2.3%)
  CWE-89: 21 (2.2%)
  CWE-1333: 19 (1.9%)
  CWE-94: 19 (1.9%)
  CWE-863: 15 (1.5%)
  CWE-287: 15 (1.5%)
  CWE-285: 15 (1.5%)
  CWE-269: 13 (1.3%)
  CWE-122: 13 (1.3%)
  CWE-502: 13 (1.3%)
  CWE-532: 12 (1.2%)
  n/a: 11 (1.1%)
  CWE-78: 11 (1.1%)
  CWE-352: 10 (1.0%)
  CWE-74: 10 (1.0%)
  CWE-416: 10 (1.0%)
  CWE-639: 9 (0.9%)
  CWE-116: 8 (0.8%)
  CWE-347: 8 (0.8%)
  CWE-345: 8 (0.8%)
  CWE-862: 8 (0.8%)
  CWE-77: 8 (0.8%)
  CWE-29: 7 (0.7%)
  CWE-248: 7 (0.7%)
  CWE-328: 7 (0.7%)
  CWE-601: 7 (0.7%)
  CWE-125: 6 (0.6%)
  CWE-61: 6 (0.6%)
  CWE-613: 6 (0.6%)
  CWE-670: 6 (0.6%)
  CWE-23: 6 (0.6%)
  CWE-434: 6 (0.6%)
  CWE-80: 6 (0.6%)
  CWE-362: 6 (0.6%)
  CWE-367: 6 (0.6%)
  CWE-1321: 5 (0.5%)
  CWE-276: 5 (0.5%)
  CWE-1336: 5 (0.5%)
  CWE-476: 5 (0.5%)
  CWE-117: 5 (0.5%)
  CWE-787: 5 (0.5%)
  CWE-312: 5 (0.5%)
  CWE-11

In [87]:
# cwes by type
cwe_types = json.load(open('cwe_types.json'))
for cwe_type in cwe_types:
    cwe_sum = sum(cwes[cwe] for cwe in cwe_types[cwe_type])
    print(f"{cwe_type}: {cwe_types[cwe_type]} ({cwe_sum/total_cwes*100:.1f}%)")
print()

Injection_and_Improper_Neutralization: ['CWE-79', 'CWE-89', 'CWE-78', 'CWE-94', 'CWE-74', 'CWE-22', 'CWE-20', 'CWE-502'] (28.9%)
Authorization_and_Authentication: ['CWE-284', 'CWE-863', 'CWE-287', 'CWE-285', 'CWE-269', 'CWE-352'] (10.7%)
Information_Exposure_and_Path_Traversal: ['CWE-200', 'CWE-532'] (5.6%)
Resource_Management_and_Denial_of_Service_DOS: ['CWE-400', 'CWE-770', 'CWE-1333'] (8.1%)
Memory_Safety_and_Allocation: ['CWE-122', 'CWE-416'] (2.4%)
Request_Forgery: ['CWE-918'] (2.3%)



In [59]:
# other cves that failed
df = pd.read_csv('large_scale/exp-3/results.csv')
failed_cves = []
for index, row in df.iterrows():
    if row['SUCCESS'] == False:
        if 'Cost' not in row['REASON'] and 'Time' not in row['REASON']:
            failed_cves.append(row['CVE'])
len(failed_cves)

203

In [64]:
# what were the type of failures for these cves
reasons = {}
for index, row in df.iterrows():
    if row['CVE'] in failed_cves:
        if 'Error code' in row['REASON']:
            reasons['Error code'] = reasons.get('Error code', []) + [row['CVE']]
        else:
            reasons[row['REASON']] = reasons.get(row['REASON'], []) + [row['CVE']]
for reason in reasons:
    print(f"{reason}: {len(reasons[reason])}")

Repo could not be built: 103
Exploiter failed: 39
CTF Verifier failed: 14
Not possible to build the repo!!!: 7
'NoneType' object is not subscriptable: 12
Error code: 25
Empty Response From LLM: 1
Connection error.: 2


In [66]:
# what are the types of projects in each category of failures
data = json.load(open('data.json'))
project_types = json.load(open('project_types.json'))
for reason in reasons:
    print(f"{reason}:")
    types_count = {}
    for cve in reasons[reason]:
        patch_url = data[cve]['patch_commits'][0]['url'].split('/')
        project_name = patch_url[-4] + '/' + patch_url[-3]
        project_type = [pt for pt in project_types if project_name in project_types[pt]][0]
        types_count[project_type] = types_count.get(project_type, 0) + 1
    for project_type in sorted(types_count, key=lambda x: types_count[x], reverse=True):
        print(f"  {project_type}: {types_count[project_type]}")
    print()

Repo could not be built:
  Web Application/Backend: 53
  Library/Framework: 12
  Cloud/DevOps/Orchestration: 11
  Operating System/Runtime: 7
  Blockchain/Crypto/FinTech: 6
  Desktop Application: 5
  Embedded/Networking: 3
  AI/ML Platform: 3
  CLI Tool/Utility: 2
  Mobile Application/SDK: 1

Exploiter failed:
  Web Application/Backend: 16
  Library/Framework: 12
  CLI Tool/Utility: 2
  Desktop Application: 2
  Operating System/Runtime: 2
  Security Tool/Server: 2
  Embedded/Networking: 1
  AI/ML Platform: 1
  Blockchain/Crypto/FinTech: 1

CTF Verifier failed:
  Web Application/Backend: 4
  Library/Framework: 4
  CLI Tool/Utility: 2
  Embedded/Networking: 1
  Operating System/Runtime: 1
  Security Tool/Server: 1
  Cloud/DevOps/Orchestration: 1

Not possible to build the repo!!!:
  Desktop Application: 3
  Library/Framework: 2
  CLI Tool/Utility: 1
  Web Application/Backend: 1

'NoneType' object is not subscriptable:
  Web Application/Backend: 6
  Library/Framework: 2
  Desktop Applicat

In [68]:
# what are the types of advisories in each category of failures
data = json.load(open('data.json'))
for reason in reasons:
    print(f"{reason}:")
    eff, ineff = 0, 0
    for cve in reasons[reason]:
        if 'sec_adv' in data[cve] and data[cve]['sec_adv']:
            if any(adv['effective'] for adv in data[cve]['sec_adv']):
                eff += 1
            else:
                ineff += 1
    print(f"Effective: {eff}, Ineffective: {ineff}")
    print()

Repo could not be built:
Effective: 44, Ineffective: 55

Exploiter failed:
Effective: 19, Ineffective: 16

CTF Verifier failed:
Effective: 5, Ineffective: 9

Not possible to build the repo!!!:
Effective: 4, Ineffective: 3

'NoneType' object is not subscriptable:
Effective: 7, Ineffective: 5

Error code:
Effective: 12, Ineffective: 12

Empty Response From LLM:
Effective: 0, Ineffective: 1

Connection error.:
Effective: 1, Ineffective: 1



In [91]:
# what are the types of cwes in each category of failures
data = json.load(open('data.json'))
for reason in reasons:
    print(f"{reason}:")
    cwes = {}
    for cve in reasons[reason]:
        if 'cwe' in data[cve] and data[cve]['cwe']:
            for cwe in data[cve]['cwe']:
                cwes[cwe['id']] = cwes.get(cwe['id'], 0) + 1
    for cwe in sorted(cwes, key=lambda x: cwes[x], reverse=True):
        print(f"  {cwe}: {cwes[cwe]}")
    print()

Repo could not be built:
  CWE-79: 12
  CWE-200: 10
  CWE-284: 6
  CWE-89: 6
  CWE-863: 6
  CWE-400: 5
  CWE-22: 4
  CWE-209: 3
  CWE-918: 3
  CWE-502: 3
  CWE-20: 2
  CWE-670: 2
  CWE-328: 2
  CWE-639: 2
  CWE-434: 2
  CWE-116: 2
  CWE-476: 2
  CWE-404: 2
  CWE-59: 1
  CWE-252: 1
  CWE-307: 1
  CWE-682: 1
  CWE-115: 1
  CWE-248: 1
  CWE-266: 1
  CWE-327: 1
  CWE-706: 1
  CWE-922: 1
  CWE-691: 1
  CWE-113: 1
  CWE-426: 1
  CWE-74: 1
  CWE-359: 1
  CWE-862: 1
  CWE-379: 1
  CWE-770: 1
  CWE-287: 1
  CWE-362: 1
  CWE-201: 1
  CWE-172: 1
  CWE-281: 1
  CWE-117: 1
  CWE-324: 1
  CWE-696: 1
  CWE-460: 1
  CWE-122: 1
  CWE-285: 1
  CWE-1287: 1
  CWE-613: 1
  CWE-294: 1
  CWE-384: 1
  CWE-204: 1
  CWE-783: 1
  CWE-290: 1
  CWE-345: 1
  CWE-78: 1
  CWE-451: 1
  CWE-125: 1
  CWE-427: 1
  CWE-94: 1
  CWE-191: 1
  CWE-787: 1
  CWE-653: 1
  CWE-521: 1

Exploiter failed:
  CWE-79: 18
  CWE-770: 3
  CWE-287: 2
  CWE-20: 1
  CWE-863: 1
  CWE-415: 1
  CWE-345: 1
  CWE-400: 1
  CWE-200: 1
  CWE-73: 1
 

### Successes

In [2]:
# what are successful cves in each run
successes = {
    'exp-1': [],
    'exp-2': [],
    'exp-3': []
}
failures = {
    'exp-1': [],
    'exp-2': [],
    'exp-3': []
}
overruns = {
    'exp-1': [],
    'exp-2': [],
    'exp-3': []
}
for exp in ['exp-1', 'exp-2', 'exp-3']:
    df = pd.read_csv(f'large_scale/{exp}/results.csv')
    for index, row in df.iterrows():
        if row['SUCCESS'] == True:
            successes[exp].append(row['CVE'])
        elif row['SUCCESS'] == False and 'Cost' not in row['REASON'] and 'Time' not in row['REASON'] and 'Error code' not in row['REASON']:
            failures[exp].append(row['CVE'])
        elif 'Cost' in row['REASON'] or 'Time' in row['REASON']:
            overruns[exp].append(row['CVE'])

print(f"Successes: {len(successes['exp-1'])}, {len(successes['exp-2'])}, {len(successes['exp-3'])}")
print(f"Failures: {len(failures['exp-1'])}, {len(failures['exp-2'])}, {len(failures['exp-3'])}")

Successes: 256, 109, 63
Failures: 294, 209, 178


In [125]:
# what are the types of projects in general, most common CWEs for each project type with effective, ineffective advisories, success and failure rates
data = json.load(open('data.json'))
project_types = json.load(open('project_types.json'))
project_cve_map = {pt: {'count': 0, 'success': 0, 'distinct_projects': len(project_types[pt]), 'success_projects': set(), 'cwes': {}} for pt in project_types}
for cve in data:
    if 'patch_commits' not in data[cve] or not data[cve]['patch_commits']:
        continue
    patch_url = data[cve]['patch_commits'][0]['url'].split('/')
    project_name = patch_url[-4] + '/' + patch_url[-3]
    for pt in project_types:
        if project_name in project_types[pt]:
            project_cve_map[pt]['count'] += 1
            if cve in successes['exp-1'] or cve in successes['exp-2'] or cve in successes['exp-3']:
                project_cve_map[pt]['success'] += 1
                project_cve_map[pt]['success_projects'].add(project_name)
            if 'cwe' in data[cve] and data[cve]['cwe']:
                for cwe in data[cve]['cwe']:
                    if cwe['id'] not in project_cve_map[pt]['cwes']:
                        project_cve_map[pt]['cwes'][cwe['id']] = {'count': 0, 'effective': {'success': 0, 'failure': 0}, 'ineffective': {'success': 0, 'failure': 0}}
                    project_cve_map[pt]['cwes'][cwe['id']]['count'] += 1
                    if any(adv['effective'] for adv in data[cve]['sec_adv']):
                        if cve in successes['exp-1'] or cve in successes['exp-2'] or cve in successes['exp-3']:
                            project_cve_map[pt]['cwes'][cwe['id']]['effective']['success'] += 1
                        else:
                            project_cve_map[pt]['cwes'][cwe['id']]['effective']['failure'] += 1
                    else:
                        if cve in successes['exp-1'] or cve in successes['exp-2'] or cve in successes['exp-3']:
                            project_cve_map[pt]['cwes'][cwe['id']]['ineffective']['success'] += 1
                        else:
                            project_cve_map[pt]['cwes'][cwe['id']]['ineffective']['failure'] += 1
for project_type in project_cve_map:
    print(f"{project_type}: [{project_cve_map[project_type]['success']} / {project_cve_map[project_type]['count']} ({project_cve_map[project_type]['success']/project_cve_map[project_type]['count']*100:.1f}% CVEs)] -- [{len(project_cve_map[project_type]['success_projects'])} / {project_cve_map[project_type]['distinct_projects']} ({len(project_cve_map[project_type]['success_projects'])/project_cve_map[project_type]['distinct_projects']*100:.1f}% Projects)]")
    for cwe in sorted(project_cve_map[project_type]['cwes'], key=lambda x: project_cve_map[project_type]['cwes'][x]['count'], reverse=True)[:10]:
        print(f"    {cwe}: {project_cve_map[project_type]['cwes'][cwe]['count']} -- (Eff: [S: {project_cve_map[project_type]['cwes'][cwe]['effective']['success']} - F: {project_cve_map[project_type]['cwes'][cwe]['effective']['failure']}], Ineff: [S: {project_cve_map[project_type]['cwes'][cwe]['ineffective']['success']} - F: {project_cve_map[project_type]['cwes'][cwe]['ineffective']['failure']}])")
    print()

AI/ML Platform: [22 / 38 (57.9% CVEs)] -- [14 / 16 (87.5% Projects)]
    CWE-502: 6 -- (Eff: [S: 3 - F: 1], Ineff: [S: 1 - F: 1])
    CWE-29: 3 -- (Eff: [S: 2 - F: 1], Ineff: [S: 0 - F: 0])
    CWE-20: 3 -- (Eff: [S: 1 - F: 2], Ineff: [S: 0 - F: 0])
    CWE-77: 3 -- (Eff: [S: 0 - F: 2], Ineff: [S: 1 - F: 0])
    CWE-94: 2 -- (Eff: [S: 0 - F: 1], Ineff: [S: 0 - F: 1])
    CWE-89: 2 -- (Eff: [S: 2 - F: 0], Ineff: [S: 0 - F: 0])
    CWE-918: 2 -- (Eff: [S: 2 - F: 0], Ineff: [S: 0 - F: 0])
    CWE-23: 2 -- (Eff: [S: 1 - F: 1], Ineff: [S: 0 - F: 0])
    CWE-400: 2 -- (Eff: [S: 1 - F: 0], Ineff: [S: 1 - F: 0])
    CWE-78: 2 -- (Eff: [S: 1 - F: 1], Ineff: [S: 0 - F: 0])

Web Application/Backend: [171 / 383 (44.6% CVEs)] -- [86 / 156 (55.1% Projects)]
    CWE-79: 99 -- (Eff: [S: 34 - F: 37], Ineff: [S: 10 - F: 18])
    CWE-284: 29 -- (Eff: [S: 12 - F: 8], Ineff: [S: 2 - F: 7])
    CWE-22: 21 -- (Eff: [S: 7 - F: 6], Ineff: [S: 3 - F: 5])
    CWE-200: 19 -- (Eff: [S: 5 - F: 3], Ineff: [S: 2 - F:

In [107]:
# what are the types of projects in each successful run and most common CWEs for each project type
data = json.load(open('data.json'))
project_types = json.load(open('project_types.json'))
successes_type_counts = {}
for exp in successes:
    print(f"{exp}:")
    types_count = {pt: {'count': 0, 'cwes': {}} for pt in project_types}
    for cve in successes[exp]:
        if 'patch_commits' not in data[cve] or not data[cve]['patch_commits']:
            continue
        patch_url = data[cve]['patch_commits'][0]['url'].split('/')
        project_name = patch_url[-4] + '/' + patch_url[-3]
        project_type = [pt for pt in project_types if project_name in project_types[pt]][0]
        types_count[project_type]['count'] += 1
        if 'cwe' in data[cve] and data[cve]['cwe']:
            for cwe in data[cve]['cwe']:
                if cwe['id'] not in types_count[project_type]['cwes']:
                    types_count[project_type]['cwes'][cwe['id']] = {'count': 0, 'effective': 0, 'ineffective': 0}
                types_count[project_type]['cwes'][cwe['id']]['count'] += 1
                if any(adv['effective'] for adv in data[cve]['sec_adv']):
                    types_count[project_type]['cwes'][cwe['id']]['effective'] += 1
                else:
                    types_count[project_type]['cwes'][cwe['id']]['ineffective'] += 1
    for project_type in types_count:
        print(f"  {project_type}: {types_count[project_type]['count']}")
        for cwe in sorted(types_count[project_type]['cwes'], key=lambda x: types_count[project_type]['cwes'][x]['count'], reverse=True)[:10]:
            print(f"    {cwe}: {types_count[project_type]['cwes'][cwe]['count']} -- (Eff: {types_count[project_type]['cwes'][cwe]['effective']}, Ineff: {types_count[project_type]['cwes'][cwe]['ineffective']})")
    print('-'*100)
    successes_type_counts[exp] = types_count

exp-1:
  AI/ML Platform: 10
    CWE-502: 3 -- (Eff: 2, Ineff: 1)
    CWE-248: 1 -- (Eff: 1, Ineff: 0)
    CWE-29: 1 -- (Eff: 1, Ineff: 0)
    CWE-400: 1 -- (Eff: 1, Ineff: 0)
    CWE-755: 1 -- (Eff: 1, Ineff: 0)
    CWE-89: 1 -- (Eff: 1, Ineff: 0)
    CWE-23: 1 -- (Eff: 1, Ineff: 0)
    CWE-77: 1 -- (Eff: 0, Ineff: 1)
  Web Application/Backend: 106
    CWE-79: 24 -- (Eff: 19, Ineff: 5)
    CWE-284: 12 -- (Eff: 12, Ineff: 0)
    CWE-200: 6 -- (Eff: 4, Ineff: 2)
    CWE-89: 6 -- (Eff: 5, Ineff: 1)
    CWE-285: 5 -- (Eff: 3, Ineff: 2)
    CWE-22: 5 -- (Eff: 3, Ineff: 2)
    CWE-400: 4 -- (Eff: 4, Ineff: 0)
    CWE-862: 4 -- (Eff: 3, Ineff: 1)
    CWE-918: 4 -- (Eff: 3, Ineff: 1)
    CWE-287: 4 -- (Eff: 4, Ineff: 0)
  Library/Framework: 101
    CWE-79: 12 -- (Eff: 4, Ineff: 8)
    CWE-400: 9 -- (Eff: 4, Ineff: 5)
    CWE-200: 5 -- (Eff: 5, Ineff: 0)
    CWE-1333: 5 -- (Eff: 5, Ineff: 0)
    CWE-770: 5 -- (Eff: 3, Ineff: 2)
    CWE-22: 5 -- (Eff: 3, Ineff: 2)
    CWE-347: 4 -- (Eff: 3, Inef

In [108]:
# what are the types of advisories in each failed run and most common CWEs for each project type
data = json.load(open('data.json'))
project_types = json.load(open('project_types.json'))
failures_type_counts = {}
for exp in failures:
    print(f"{exp}:")
    types_count = {pt: {'count': 0, 'cwes': {}} for pt in project_types}
    for cve in failures[exp]:
        if 'patch_commits' not in data[cve] or not data[cve]['patch_commits']:
            continue
        patch_url = data[cve]['patch_commits'][0]['url'].split('/')
        project_name = patch_url[-4] + '/' + patch_url[-3]
        project_type = [pt for pt in project_types if project_name in project_types[pt]][0]
        types_count[project_type]['count'] += 1
        if 'cwe' in data[cve] and data[cve]['cwe']:
            for cwe in data[cve]['cwe']:
                if cwe['id'] not in types_count[project_type]['cwes']:
                    types_count[project_type]['cwes'][cwe['id']] = {'count': 0, 'effective': 0, 'ineffective': 0}
                types_count[project_type]['cwes'][cwe['id']]['count'] += 1
                if any(adv['effective'] for adv in data[cve]['sec_adv']):
                    types_count[project_type]['cwes'][cwe['id']]['effective'] += 1
                else:
                    types_count[project_type]['cwes'][cwe['id']]['ineffective'] += 1
    for project_type in types_count:
        print(f"  {project_type}: {types_count[project_type]['count']}")
        for cwe in sorted(types_count[project_type]['cwes'], key=lambda x: types_count[project_type]['cwes'][x]['count'], reverse=True)[:10]:
            print(f"    {cwe}: {types_count[project_type]['cwes'][cwe]['count']} -- (Eff: {types_count[project_type]['cwes'][cwe]['effective']}, Ineff: {types_count[project_type]['cwes'][cwe]['ineffective']})")
    print('-'*100)
    failures_type_counts[exp] = types_count

exp-1:
  AI/ML Platform: 12
    CWE-918: 2 -- (Eff: 2, Ineff: 0)
    CWE-20: 1 -- (Eff: 1, Ineff: 0)
    CWE-1336: 1 -- (Eff: 0, Ineff: 1)
    CWE-77: 1 -- (Eff: 1, Ineff: 0)
    CWE-94: 1 -- (Eff: 0, Ineff: 1)
    CWE-74: 1 -- (Eff: 0, Ineff: 1)
    CWE-78: 1 -- (Eff: 1, Ineff: 0)
    CWE-22: 1 -- (Eff: 1, Ineff: 0)
    CWE-502: 1 -- (Eff: 0, Ineff: 1)
    CWE-400: 1 -- (Eff: 0, Ineff: 1)
  Web Application/Backend: 112
    CWE-79: 30 -- (Eff: 23, Ineff: 7)
    CWE-22: 10 -- (Eff: 6, Ineff: 4)
    CWE-284: 8 -- (Eff: 6, Ineff: 2)
    CWE-918: 6 -- (Eff: 2, Ineff: 4)
    CWE-770: 4 -- (Eff: 1, Ineff: 3)
    CWE-200: 4 -- (Eff: 1, Ineff: 3)
    CWE-400: 4 -- (Eff: 2, Ineff: 2)
    CWE-89: 4 -- (Eff: 1, Ineff: 3)
    CWE-502: 3 -- (Eff: 0, Ineff: 3)
    CWE-20: 2 -- (Eff: 2, Ineff: 0)
  Library/Framework: 81
    CWE-79: 18 -- (Eff: 10, Ineff: 8)
    CWE-1333: 8 -- (Eff: 4, Ineff: 4)
    CWE-770: 8 -- (Eff: 2, Ineff: 6)
    CWE-200: 5 -- (Eff: 2, Ineff: 3)
    CWE-287: 3 -- (Eff: 0, Ineff:

In [ ]:
# make a table for side by side comparison of successful and failed runs using `successes_type_counts` and `failures_type_counts`
# Create a comprehensive comparison table showing CWEs and effectiveness for each experiment
print("CWE Analysis by Experiment - Successes vs Failures")
print("="*120)

# Get all experiments
all_experiments = sorted(set(successes_type_counts.keys()) | set(failures_type_counts.keys()))

for project_type in sorted(set().union(*[exp_data.keys() for exp_data in list(successes_type_counts.values()) + list(failures_type_counts.values())])):
    print(f"\n{project_type.upper()}:")
    print("-" * 100)
    
    # Collect all CWEs for this project type across all experiments
    all_cwes = set()
    for exp in all_experiments:
        if exp in successes_type_counts and project_type in successes_type_counts[exp]:
            all_cwes.update(successes_type_counts[exp][project_type]['cwes'].keys())
        if exp in failures_type_counts and project_type in failures_type_counts[exp]:
            all_cwes.update(failures_type_counts[exp][project_type]['cwes'].keys())
    
    if not all_cwes:
        print("  No CWEs found for this project type")
        continue
    
    # Create header
    header = f"{'CWE':<15}"
    for exp in all_experiments:
        header += f"{exp:<25}"
    print(header)
    print("-" * len(header))
    
    # Print data for each CWE
    for cwe in sorted(all_cwes):
        row = f"{cwe:<15}"
        for exp in all_experiments:
            # Get success data
            success_data = ""
            if (exp in successes_type_counts and 
                project_type in successes_type_counts[exp] and 
                cwe in successes_type_counts[exp][project_type]['cwes']):
                s_data = successes_type_counts[exp][project_type]['cwes'][cwe]
                success_data = f"S:{s_data['count']}(E:{s_data['effective']},I:{s_data['ineffective']})"
            
            # Get failure data
            failure_data = ""
            if (exp in failures_type_counts and 
                project_type in failures_type_counts[exp] and 
                cwe in failures_type_counts[exp][project_type]['cwes']):
                f_data = failures_type_counts[exp][project_type]['cwes'][cwe]
                failure_data = f"F:{f_data['count']}(E:{f_data['effective']},I:{f_data['ineffective']})"
            
            # Combine success and failure data
            cell_data = ""
            if success_data and failure_data:
                cell_data = f"{success_data}|{failure_data}"
            elif success_data:
                cell_data = success_data
            elif failure_data:
                cell_data = failure_data
            else:
                cell_data = "-"
            
            row += f"{cell_data:<25}"
        print(row)

print("\n" + "="*120)
print("Legend: S=Success, F=Failure, E=Effective, I=Ineffective")
print("Format: S:count(E:effective,I:ineffective)|F:count(E:effective,I:ineffective)")


CWE Analysis by Experiment - Successes vs Failures

AI/ML PLATFORM:
----------------------------------------------------------------------------------------------------
CWE            exp-1                    exp-2                    exp-3                    
------------------------------------------------------------------------------------------
CWE-115        -                        F:1(E:1,I:0)             -                        
CWE-1333       -                        S:1(E:0,I:1)             -                        
CWE-1336       F:1(E:0,I:1)             S:1(E:0,I:1)             -                        
CWE-20         F:1(E:1,I:0)             F:1(E:1,I:0)             S:1(E:1,I:0)             
CWE-22         F:1(E:1,I:0)             F:1(E:1,I:0)             S:1(E:1,I:0)             
CWE-23         S:1(E:1,I:0)             -                        -                        
CWE-248        S:1(E:1,I:0)             -                        -                        
CWE-29      

## CWEs Analysis

In [5]:
# For all successfully reproduced CVEs across experiments, list only CWEs success rates
data = json.load(open('data.json'))
cwes = {}
for cve in data:
    eff = False
    if any(adv['effective'] for adv in data[cve]['sec_adv']):
        eff = True           
    if 'cwe' in data[cve] and data[cve]['cwe']:
        for cwe in data[cve]['cwe']:
            if cwe['id'] not in cwes:
                cwes[cwe['id']] = {'total': 0, 'effective': 0, 'ineffective': 0}
            cwes[cwe['id']]['total'] += 1
            if eff:
                cwes[cwe['id']]['effective'] += 1
            else:
                cwes[cwe['id']]['ineffective'] += 1

successful_cves = set(successes['exp-1']) | set(successes['exp-2']) | set(successes['exp-3'])
cwe_success_counts = {}
for cve in successful_cves:
    eff = False
    if any(adv['effective'] for adv in data[cve]['sec_adv']):
        eff = True
    if 'cwe' in data[cve] and data[cve]['cwe']:
        for cwe in data[cve]['cwe']:
            if cwe['id'] not in cwe_success_counts:
                cwe_success_counts[cwe['id']] = {'effective': 0, 'ineffective': 0}
            if eff:
                cwe_success_counts[cwe['id']]['effective'] += 1
            else:
                cwe_success_counts[cwe['id']]['ineffective'] += 1

print("CWE Success Rates:")
for cwe in sorted(cwes, key=lambda x: cwes[x]['total'], reverse=True):
    success_count = cwe_success_counts.get(cwe, {'effective': 0})['effective'] + cwe_success_counts.get(cwe, {'ineffective': 0})['ineffective']
    total_count = cwes[cwe]['total']
    success_rate = (success_count / total_count) * 100 if total_count > 0 else 0
    print(f"  {cwe}: {success_count}/{total_count} ({success_rate:.1f}%) -- (Eff: {cwe_success_counts.get(cwe, {'effective': 0})['effective']}/{cwes[cwe]['effective']}, Ineff: {cwe_success_counts.get(cwe, {'ineffective': 0})['ineffective']}/{cwes[cwe]['ineffective']})")

CWE Success Rates:
  CWE-79: 65/142 (45.8%) -- (Eff: 43/94, Ineff: 22/48)
  CWE-200: 20/43 (46.5%) -- (Eff: 15/20, Ineff: 5/23)
  CWE-22: 16/37 (43.2%) -- (Eff: 11/22, Ineff: 5/15)
  CWE-284: 19/36 (52.8%) -- (Eff: 16/26, Ineff: 3/10)
  CWE-400: 22/36 (61.1%) -- (Eff: 13/19, Ineff: 9/17)
  CWE-20: 13/29 (44.8%) -- (Eff: 6/15, Ineff: 7/14)
  CWE-770: 14/24 (58.3%) -- (Eff: 7/9, Ineff: 7/15)
  CWE-918: 7/22 (31.8%) -- (Eff: 6/13, Ineff: 1/9)
  CWE-89: 11/21 (52.4%) -- (Eff: 8/15, Ineff: 3/6)
  CWE-1333: 15/19 (78.9%) -- (Eff: 8/10, Ineff: 7/9)
  CWE-94: 9/19 (47.4%) -- (Eff: 2/8, Ineff: 7/11)
  CWE-863: 7/15 (46.7%) -- (Eff: 4/6, Ineff: 3/9)
  CWE-287: 8/15 (53.3%) -- (Eff: 4/4, Ineff: 4/11)
  CWE-285: 7/15 (46.7%) -- (Eff: 5/8, Ineff: 2/7)
  CWE-269: 7/13 (53.8%) -- (Eff: 4/8, Ineff: 3/5)
  CWE-122: 8/13 (61.5%) -- (Eff: 2/4, Ineff: 6/9)
  CWE-502: 6/13 (46.2%) -- (Eff: 3/6, Ineff: 3/7)
  CWE-532: 6/12 (50.0%) -- (Eff: 2/2, Ineff: 4/10)
  n/a: 6/11 (54.5%) -- (Eff: 0/0, Ineff: 6/11)
  C

### Sample

In [128]:
timeout_cves = []
for reason in timeout_reason:
    timeout_cves.extend(timeout_reason[reason])

cost_overrun_cves = []
for reason in cost_overrun_reason:
    cost_overrun_cves.extend(cost_overrun_reason[reason])

len(timeout_cves), len(cost_overrun_cves)

(42, 156)

In [129]:
import random
random.seed(42)
sample_cves = random.sample(timeout_cves, 10) + random.sample(cost_overrun_cves, 20)
print(sample_cves)

['CVE-2024-52309', 'CVE-2025-30205', 'CVE-2024-51738', 'CVE-2024-0397', 'CVE-2024-43374', 'CVE-2024-34361', 'CVE-2025-32032', 'CVE-2025-25202', 'CVE-2024-0640', 'CVE-2024-53274', 'CVE-2024-8898', 'CVE-2024-8581', 'CVE-2024-52296', 'CVE-2025-27100', 'CVE-2025-32444', 'CVE-2025-24802', 'CVE-2025-29922', 'CVE-2024-35227', 'CVE-2025-27090', 'CVE-2025-46571', 'CVE-2025-27407', 'CVE-2024-12580', 'CVE-2025-27157', 'CVE-2024-47830', 'CVE-2025-4516', 'CVE-2024-8947', 'CVE-2024-2548', 'CVE-2024-39320', 'CVE-2024-43397', 'CVE-2024-51752']
